# 数据预处理
**原始脏数据——>模型能看懂、能稳定训练的数据**
几乎所有表格类竞赛、市场调查数据集第一步都是它；XGBoost、LR、SVM、KNN 对预处理要求差别很大。

## 完整流程总览
1. 导入库
2. 读取数据集
3. 分离特征X和标签Y
4. 处理缺失值
5. 分类特征编码(文字转数字)
6. 划分训练集、测试集
7. 特征缩放(标准化/归一化)
8. 【可选】异常值、重复值、特征筛选

## 导入依赖

In [2]:
# 导入numpy，数值计算库，用来处理数组、矩阵运算
import numpy as np
# 导入pandas，表格数据处理库，读取csv、处理DataFrame
import pandas as pd

# 数据集划分：把数据拆成训练集、测试集
from sklearn.model_selection import train_test_split

# 缺失值填充器：填补数据里的NaN空值
from sklearn.impute import SimpleImputer

# OneHotEncoder：独热编码，处理无序分类特征（如颜色、城市）
# LabelEncoder：标签编码，一般只用于标签y，把文本标签转为数字
# StandardScaler：标准化，就是笔记里讲的 z=(x-μ)/σ
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler

# ColumnTransformer：按列分别做不同预处理（一部分列标准化、一部分列独热编码）
from sklearn.compose import ColumnTransformer

# Pipeline：流水线，把多个预处理步骤串起来，一步fit、一步predict，防止数据泄露
from sklearn.pipeline import Pipeline


## 读入数据，区分X/y
x：特征，用来预测
y：标签，我们要预测的目标

In [4]:
dataset = pd.read_csv("./Data.csv")
# 所有行，除最后一列
X = dataset.iloc[:, :-1].values
# 最后一列
y = dataset.iloc[:, -1].values
print(X)
print(y)

[['France' 44 72000.0]
 ['Spain' 27 48000.0]
 ['Germany' 30 54000.0]
 ['France' 38 61000.0]
 ['Germany' 40 nan]]
['No' 'Yes' 'No' 'No' 'Yes']


## 缺失值处理
常见：年龄为空、薪资为空，也就是NaN
不能直接丢(样本少的时候会损失信息)，两种方案：
* 数值型：均值/中位数填充(中位数对异常值更稳健)
* 类别型：填充众数

In [5]:
# sklearn
# 只对第1、2列（Age、Salary，数值列）填充均值
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
imputer.fit(X[:, 1:3])
X[:, 1:3] = imputer.transform(X[:, 1:3])
print(X)
# ⚠️ 注意：fit 只用在训练集！不要拿测试集的均值填充训练集，会数据泄露。
# 后面用 Pipeline 可以自动规避这个坑。

[['France' 44.0 72000.0]
 ['Spain' 27.0 48000.0]
 ['Germany' 30.0 54000.0]
 ['France' 38.0 61000.0]
 ['Germany' 40.0 58750.0]]


## 分类特征编码
两种分类变量：
* 类别特征(特征里的文字)——>OneHot编码(独热)
* 标签(要预测的)——>LabelEncoder

In [6]:
# 对第0列Country做独热编码
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [0])], remainder='passthrough')
X = np.array(ct.fit_transform(X))

# 标签编码
le = LabelEncoder()
y = le.fit_transform(y)
print(X)
print(y)
# ✅ 补充：树模型（XGBoost、LightGBM）**原生支持类别，很多时候不需要 OneHot**；
# 但 KNN、SVM、线性回归必须处理。

[[1.0 0.0 0.0 44.0 72000.0]
 [0.0 0.0 1.0 27.0 48000.0]
 [0.0 1.0 0.0 30.0 54000.0]
 [1.0 0.0 0.0 38.0 61000.0]
 [0.0 1.0 0.0 40.0 58750.0]]
[0 1 0 0 1]


## 划分训练集 and 测试集
核心思想：用训练集训练模型，测试集模拟未知新数据，评估泛化能力

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, # 20% 当测试，80% 训练
    random_state=42 # 固定随机种子，保证每次划分结果一样，方便复现
)
print(X_train)
print(X_test)
print(y_train)
print(y_test)
# ✅ 重要顺序：先划分，再缩放 / 填充！ 防止数据泄露。
# ❌ 错误做法：全数据集先算均值、方差、缩放，再分割。

[[0.0 1.0 0.0 40.0 58750.0]
 [0.0 1.0 0.0 30.0 54000.0]
 [1.0 0.0 0.0 44.0 72000.0]
 [1.0 0.0 0.0 38.0 61000.0]]
[[0.0 0.0 1.0 27.0 48000.0]]
[1 0 0 0]
[1]


## 特征缩放（标准化 / 归一化）
目的：让不同量纲的特征在同一个尺度上

例子：Age 20~60，Salary 40000~100000，Salary 数值天然很大，KNN、SVM、梯度下降会被它主导。

两种：
1. StandardScaler 标准化（最常用）
$$z = \frac{x-\mu}{\sigma}$$均值 0，方差 1

👍 适合：SVM、KNN、逻辑回归、神经网络
👍 树模型（RF/XGB/LGB）不需要！对尺度不敏感 
2. MinMaxScaler 归一化：缩放到 [0,1]，离群点敏感，现在用得少。

In [8]:
sc = StandardScaler()
# 只用训练集拟合均值方差
X_train[:, 3:] = sc.fit_transform(X_train[:, 3:])
# 测试集只 transform，不 fit！
X_test[:, 3:] = sc.transform(X_test[:, 3:])
print(X_train)
print(X_test)

[[0.0 1.0 0.0 0.3922322702763681 -0.40713052783664927]
 [0.0 1.0 0.0 -1.5689290811054724 -1.1267100654084015]
 [1.0 0.0 0.0 1.1766968108291043 1.6001176559161332]
 [1.0 0.0 0.0 0.0 -0.06627706267108244]]
[[0.0 0.0 1.0 -2.1572774865200244 -2.035652639183246]]
